# 🏊 Object Pool Design Pattern — Hinglish mein samjho (Beginner Friendly)

Is notebook mein hum **Object Pool Design Pattern** ko step-by-step, Hinglish comments ke saath samjhenge.
Har code cell ke andar ek **multiline comment (docstring)** hai jisme concept Hinglish mein explain kiya gaya hai.

Pehle dekhenge **bina pattern ke** kya problem aati hai, phir **Object Pool Pattern** se use solve karenge.
Aakhir mein ek **Practice Question** aur uska **Coding Solution** bhi diya gaya hai taaki tum khud practice kar sako.

> Note: Original documentation mein example Java mein tha (database connections ke liye) — humne wahi concept Python mein implement kiya hai taaki samajhna aasan ho.

In [1]:
"""
OBJECT POOL DESIGN PATTERN — KYA HAI?
==========================================

Object Pool ek CREATIONAL design pattern hai jo REUSABLE objects
ka ek "POOL" (jama-khaata / storage) maintain karta hai, taaki
baar baar EXPENSIVE objects CREATE aur DESTROY na karne pade.

SOCHO EK EXAMPLE:
--------------------
Ek web application mein HAR user request database se baat karti
hai (data fetch karna, order save karna, etc). Lekin HAR request
ke liye NAYA database connection banana EXPENSIVE hai (time
lagta hai, resources use hote hain).

Object Pool isko solve karta hai:
    -> Kuch connections PEHLE SE bana ke POOL mein rakh do
    -> Client jab connection maange, POOL se ek "BORROW" (udhaar)
       kar le -- naya banane ki zaroorat nahi
    -> Kaam ho jaane ke baad connection wapas POOL mein "RETURN"
       kar do, taaki koi aur use kar sake

Isse HAMESHA thodi si limited connections REUSE hote rehte hain,
scratch se naya banane ka overhead nahi lagta.

MAIN COMPONENTS:
--------------------
1. PooledObject (Reusable Object) -> Wo object jo expensive hai
                                     banane mein (e.g. DB Connection)
2. ObjectPool                       -> Sabse important class -- ye
                                     AVAILABLE (free) aur IN-USE
                                     objects ka track rakhti hai
3. Client                            -> Pool se object BORROW karta
                                     hai, use karta hai, aur wapas
                                     RETURN kar deta hai

OBJECT POOL LIFECYCLE (5 STAGES):
--------------------------------------
1. Creation    -> Objects banaye jaate hain aur pool mein daale jaate hain
2. Borrowing   -> Client pool se object "borrow" (le) leta hai
3. Usage        -> Client us object ko apne kaam ke liye use karta hai
4. Returning    -> Kaam khatam hone par object POOL mein WAPAS kar deta hai
5. Rejection/Destruction -> Agar pool FULL hai ya object EXPIRE ho gaya
                             hai, to use hata diya jaata hai

REAL LIFE EXAMPLES:
-----------------------
- Database Connection Pool: Web apps DB connections reuse karte hain
- Thread Pool: Servers threads ka pool rakhte hain requests handle
  karne ke liye
- HTTP Connection Pool: API calls ke liye connections reuse hote hain
- Game Development: Bullets, enemies, particles reuse hote hain
  (naye banane/destroy karne ke bajaye)

FAYDE (ADVANTAGES):
------------------------
- Baar baar object CREATE/DESTROY karne ka overhead kam hota hai
- Borrow/return ka mechanism milta hai, usage track hoti hai
- Resource-intensive apps mein PERFORMANCE improve hoti hai
- Memory usage OPTIMIZE hoti hai aur objects ki sankhya CONTROL mein rehti hai

NUKSAAN (DISADVANTAGES):
----------------------------
- Agar object properly RETURN nahi kiya gaya, to RESOURCE LEAK ho
  sakta hai (object hamesha ke liye "in-use" reh jaata hai)
- Multi-threaded environments mein THREAD-SAFETY issues aa sakte hain
- Halke/sasste objects ke liye ye pattern USELESS hai (overkill)
- Agar pool size bahut BADI rakhi to MEMORY USAGE badh sakta hai
"""
print("Concept samajh liya! Ab neeche code mein implement karke dekhte hain.")


Concept samajh liya! Ab neeche code mein implement karke dekhte hain.


## ❌ Pehle dekho: BINA Object Pool Pattern ke (Problem Code)

Chalo pehle ye dekhte hain ki Object Pool Pattern KE BINA log Database Connections kaise banate hain
aur destroy karte hain, aur usme kya dikkat aati hai.

In [2]:
"""
BINA OBJECT POOL KE — PROBLEM DEMO
----------------------------------------
Yahan har "user request" ke liye NAYA DatabaseConnectionBad object
banaya jaa raha hai, aur kaam khatam hote hi CLOSE (destroy) kiya
jaa raha hai. Connection banana EXPENSIVE hai (simulate kiya hai
"time.sleep()" se -- real app mein ye network/auth handshake hota hai).
"""

import time

class DatabaseConnectionBad:
    _counter = 0

    def __init__(self):
        DatabaseConnectionBad._counter += 1
        self.id = DatabaseConnectionBad._counter
        print(f"⏳ Naya DB Connection #{self.id} banaya jaa raha hai (EXPENSIVE)...")
        time.sleep(0.3)   # connection setup ka simulation (auth, handshake, etc.)

    def query(self, sql):
        print(f"[Connection #{self.id}] Running: {sql}")

    def close(self):
        print(f"❌ Connection #{self.id} CLOSE (destroy) kiya jaa raha hai")


start = time.time()

# 5 alag alag "user requests" simulate kar rahe hain
for request_num in range(1, 6):
    print(f"\n--- Request #{request_num} aaya ---")
    conn = DatabaseConnectionBad()   # HAR BAAR naya connection banaya!
    conn.query(f"SELECT * FROM users WHERE request_id={request_num}")
    conn.close()                      # kaam khatam, connection DESTROY kar diya

end = time.time()
print(f"\n⏱️ Total time liya: {end - start:.2f} seconds")



--- Request #1 aaya ---
⏳ Naya DB Connection #1 banaya jaa raha hai (EXPENSIVE)...


[Connection #1] Running: SELECT * FROM users WHERE request_id=1
❌ Connection #1 CLOSE (destroy) kiya jaa raha hai

--- Request #2 aaya ---
⏳ Naya DB Connection #2 banaya jaa raha hai (EXPENSIVE)...


[Connection #2] Running: SELECT * FROM users WHERE request_id=2
❌ Connection #2 CLOSE (destroy) kiya jaa raha hai

--- Request #3 aaya ---
⏳ Naya DB Connection #3 banaya jaa raha hai (EXPENSIVE)...


[Connection #3] Running: SELECT * FROM users WHERE request_id=3
❌ Connection #3 CLOSE (destroy) kiya jaa raha hai

--- Request #4 aaya ---
⏳ Naya DB Connection #4 banaya jaa raha hai (EXPENSIVE)...


[Connection #4] Running: SELECT * FROM users WHERE request_id=4
❌ Connection #4 CLOSE (destroy) kiya jaa raha hai

--- Request #5 aaya ---
⏳ Naya DB Connection #5 banaya jaa raha hai (EXPENSIVE)...


[Connection #5] Running: SELECT * FROM users WHERE request_id=5
❌ Connection #5 CLOSE (destroy) kiya jaa raha hai

⏱️ Total time liya: 1.50 seconds


In [3]:
"""
ISME PROBLEM KYA THI?
-------------------------
1. REPEATED EXPENSIVE CREATION: Har request ke liye NAYA connection
   banaya gaya, aur kaam khatam hote hi turant DESTROY kar diya gaya.
   5 requests ke liye upar ~1.5 second se zyada laga -- ek REAL app
   mein jahan HAZAARON requests aati hain, ye TIME bahut zyada
   accumulate ho jaayega.

2. DATABASE SERVER OVERLOAD: Agar bahut saare users EK SAATH
   requests bhejein, to HAR request apna NAYA connection kholega --
   database server par bahut zyada connections KHUL JAAYENGE, jo
   server ko SLOW ya CRASH kar sakta hai.

3. NO REUSE: Ek connection jo abhi CLOSE hua, wo PERFECTLY REUSABLE
   tha agli request ke liye -- lekin humne use FEKH diya aur PHIR
   SE naya banaya. Ye WASTE hai.

4. NO LIMIT CONTROL: Is approach mein koi CONTROL nahi hai ki
   MAXIMUM kitne connections ek saath ban sakte hain -- agar traffic
   spike ho jaaye to UNLIMITED connections ban sakte hain.

YE SAB PROBLEMS "Object Pool Design Pattern" SOLVE KARTA HAI!
Neeche BEST APPROACH dekhte hain -- connections ko POOL mein rakhenge
aur BAAR BAAR REUSE karenge, naya banane ke bajaye.
"""
print("Chalo ab BEST APPROACH (Object Pool Pattern) dekhte hain...")


Chalo ab BEST APPROACH (Object Pool Pattern) dekhte hain...


## ✅ Ab dekho: BEST APPROACH — Object Pool Pattern se solve karte hain

### 1️⃣ PooledObject — `DatabaseConnection`

In [4]:
"""
POOLED OBJECT (DatabaseConnection)
----------------------------------------
Ye wo object hai jo EXPENSIVE hai banane mein, isliye ye POOL
mein rakha jaayega aur REUSE hoga. Isme ek "in_use" flag bhi hai
jisse pool ko pata chale ki ye connection abhi kisi client ke
paas hai ya FREE hai.
"""

class DatabaseConnection:
    _counter = 0

    def __init__(self):
        DatabaseConnection._counter += 1
        self.id = DatabaseConnection._counter
        print(f"⏳ Naya DB Connection #{self.id} banaya jaa raha hai (SIRF EK BAAR)...")
        time.sleep(0.3)   # expensive setup, lekin ab ye baar baar NAHI hoga
        self.in_use = False

    def query(self, sql):
        print(f"[Connection #{self.id}] Running: {sql}")

    def close(self):
        print(f"❌ Connection #{self.id} permanently CLOSE kiya jaa raha hai")


### 2️⃣ ObjectPool — Available aur In-Use connections track karta hai

In [5]:
"""
OBJECT POOL
----------------
Ye PATTERN ka SABSE IMPORTANT class hai. Ye do lists maintain
karti hai:
    - available -> FREE connections jo abhi kisi ke paas nahi hain
    - in_use     -> connections jo abhi kisi client ke paas hain

MAIN METHODS:
    - borrow_connection() -> agar "available" mein connection hai
      to WAHI de do (reuse), warna NAYA banao (lekin max_size tak
      hi)
    - return_connection()  -> connection wapas "available" mein
      daal do taaki koi aur use kar sake
"""

class ConnectionPool:
    def __init__(self, max_size=3):
        self.max_size = max_size
        self.available = []   # free connections
        self.in_use = []      # connections jo abhi use ho rahi hain

    def borrow_connection(self):
        if self.available:
            # POOL mein pehle se free connection hai -- REUSE karo!
            conn = self.available.pop()
            print(f"♻️  Connection #{conn.id} POOL se REUSE ho raha hai (naya nahi banaya)")
        elif len(self.in_use) < self.max_size:
            # Pool khaali hai lekin abhi max_size cross nahi hua -- naya banao
            conn = DatabaseConnection()
        else:
            # Pool FULL hai -- naya connection nahi de sakte
            raise Exception("⚠️ Pool full hai! Koi connection available nahi, wait karo.")

        conn.in_use = True
        self.in_use.append(conn)
        return conn

    def return_connection(self, conn):
        conn.in_use = False
        self.in_use.remove(conn)
        self.available.append(conn)
        print(f"↩️  Connection #{conn.id} POOL mein WAPAS aa gaya (reuse ke liye ready)")


### 3️⃣ Client — Pool se borrow/return karke connections use karta hai

In [6]:
"""
CLIENT CODE
---------------
Client ko sirf itna karna hai: pool.borrow_connection() bulao,
kaam karo, aur pool.return_connection() se WAPAS de do. Client
ko ye pata NAHI hona chahiye ki andar connection NAYA bana ya
REUSE hua.
"""

pool = ConnectionPool(max_size=3)

start = time.time()

# Same 5 requests, lekin ab POOL use karke
for request_num in range(1, 6):
    print(f"\n--- Request #{request_num} aaya ---")
    conn = pool.borrow_connection()          # pool se connection LIYA
    conn.query(f"SELECT * FROM users WHERE request_id={request_num}")
    pool.return_connection(conn)              # kaam khatam, WAPAS kar diya

end = time.time()
print(f"\n⏱️ Total time liya: {end - start:.2f} seconds  (pehle wale se KAAFI kam!)")
print(f"\nTotal connections banaye gaye: {DatabaseConnection._counter} (bina pool ke to 5 bante!)")



--- Request #1 aaya ---
⏳ Naya DB Connection #1 banaya jaa raha hai (SIRF EK BAAR)...


[Connection #1] Running: SELECT * FROM users WHERE request_id=1
↩️  Connection #1 POOL mein WAPAS aa gaya (reuse ke liye ready)

--- Request #2 aaya ---
♻️  Connection #1 POOL se REUSE ho raha hai (naya nahi banaya)
[Connection #1] Running: SELECT * FROM users WHERE request_id=2
↩️  Connection #1 POOL mein WAPAS aa gaya (reuse ke liye ready)

--- Request #3 aaya ---
♻️  Connection #1 POOL se REUSE ho raha hai (naya nahi banaya)
[Connection #1] Running: SELECT * FROM users WHERE request_id=3
↩️  Connection #1 POOL mein WAPAS aa gaya (reuse ke liye ready)

--- Request #4 aaya ---
♻️  Connection #1 POOL se REUSE ho raha hai (naya nahi banaya)
[Connection #1] Running: SELECT * FROM users WHERE request_id=4
↩️  Connection #1 POOL mein WAPAS aa gaya (reuse ke liye ready)

--- Request #5 aaya ---
♻️  Connection #1 POOL se REUSE ho raha hai (naya nahi banaya)
[Connection #1] Running: SELECT * FROM users WHERE request_id=5
↩️  Connection #1 POOL mein WAPAS aa gaya (reuse ke liye ready)

⏱️ Tota

## 📝 Practice Question (Tumhari baari!)

**Problem Statement:**

Ek **Thread Pool jaisa "Worker Pool" System** banao using Object Pool Design Pattern (jaise document
mein "Thread Pool (Server Systems)" ka real-world application mention hua tha).

Har `Worker` object banana EXPENSIVE hai (setup mein time lagta hai — simulate karo `time.sleep()`
se). Ek server ko multiple "tasks" handle karne hain, lekin har task ke liye naya `Worker` banane ke
bajaye, EXISTING workers ko POOL se REUSE karna hai.

Tumhe ye banana hai (ConnectionPool example jaisi hi structure use karo):

1. `Worker` (PooledObject) — `__init__` mein expensive setup simulate karo (`time.sleep(0.2)`), ek `execute_task(task_name)` method ho jo print kare `"Worker #{id} executing: {task_name}"`
2. `WorkerPool` (ObjectPool) — `available` aur `in_use` lists, `max_size` limit, `borrow_worker()` aur `return_worker()` methods
3. Client code likho jo `WorkerPool(max_size=2)` banaye aur 4 tasks ko process kare — har task ke liye worker borrow karo, `execute_task()` call karo, phir wapas return karo
4. Verify karo ki total kitne `Worker` objects actually create hue (max 2 hone chahiye, chahe 4 tasks process hue hon)

👉 Pehle khud try karo neeche wale khaali code cell mein, phir uske baad wale cell mein solution dekho!

In [7]:
"""
YAHAN KHUD LIKHO (Practice Cell)
-----------------------------------
Hint: DatabaseConnection/ConnectionPool example ko copy karke sirf
naam badal do:
DatabaseConnection -> Worker, ConnectionPool -> WorkerPool
query() -> execute_task()

Apna code neeche likho:
"""

# TODO: class Worker banao (expensive __init__, execute_task() method)
# TODO: class WorkerPool banao (available, in_use, max_size, borrow_worker(), return_worker())
# TODO: Client code likho: pool banao, 4 tasks process karo, verify karo max 2 workers bane


'\nYAHAN KHUD LIKHO (Practice Cell)\n-----------------------------------\nHint: DatabaseConnection/ConnectionPool example ko copy karke sirf\nnaam badal do:\nDatabaseConnection -> Worker, ConnectionPool -> WorkerPool\nquery() -> execute_task()\n\nApna code neeche likho:\n'

## ✅ Solution — Practice Question ka Coding Solution

In [8]:
"""
SOLUTION EXPLANATION
-----------------------
Yahan bilkul DatabaseConnection/ConnectionPool jaisi hi structure
use ki hai, bas ab "Workers" bana rahe hain instead of "DB Connections".

1. Worker       -> PooledObject
2. WorkerPool    -> ObjectPool
3. Neeche client code hai jo 4 tasks process karta hai using max 2 workers
"""

class Worker:
    _counter = 0

    def __init__(self):
        Worker._counter += 1
        self.id = Worker._counter
        print(f"⏳ Naya Worker #{self.id} setup ho raha hai (EXPENSIVE)...")
        time.sleep(0.2)
        self.in_use = False

    def execute_task(self, task_name):
        print(f"⚙️  Worker #{self.id} executing: {task_name}")


class WorkerPool:
    def __init__(self, max_size=2):
        self.max_size = max_size
        self.available = []
        self.in_use = []

    def borrow_worker(self):
        if self.available:
            worker = self.available.pop()
            print(f"♻️  Worker #{worker.id} POOL se REUSE ho raha hai")
        elif len(self.in_use) < self.max_size:
            worker = Worker()
        else:
            raise Exception("⚠️ Pool full hai! Koi worker available nahi.")

        worker.in_use = True
        self.in_use.append(worker)
        return worker

    def return_worker(self, worker):
        worker.in_use = False
        self.in_use.remove(worker)
        self.available.append(worker)
        print(f"↩️  Worker #{worker.id} POOL mein WAPAS aa gaya")


# ---- Client Code ----
worker_pool = WorkerPool(max_size=2)

tasks = ["Resize Image", "Send Email", "Generate Report", "Backup Database"]

for task in tasks:
    print(f"\n--- Task: {task} ---")
    worker = worker_pool.borrow_worker()
    worker.execute_task(task)
    worker_pool.return_worker(worker)

print(f"\nTotal Worker objects banaye gaye: {Worker._counter} (max_size=2 tha, isse zyada KABHI nahi banenge!)")



--- Task: Resize Image ---
⏳ Naya Worker #1 setup ho raha hai (EXPENSIVE)...


⚙️  Worker #1 executing: Resize Image
↩️  Worker #1 POOL mein WAPAS aa gaya

--- Task: Send Email ---
♻️  Worker #1 POOL se REUSE ho raha hai
⚙️  Worker #1 executing: Send Email
↩️  Worker #1 POOL mein WAPAS aa gaya

--- Task: Generate Report ---
♻️  Worker #1 POOL se REUSE ho raha hai
⚙️  Worker #1 executing: Generate Report
↩️  Worker #1 POOL mein WAPAS aa gaya

--- Task: Backup Database ---
♻️  Worker #1 POOL se REUSE ho raha hai
⚙️  Worker #1 executing: Backup Database
↩️  Worker #1 POOL mein WAPAS aa gaya

Total Worker objects banaye gaye: 1 (max_size=2 tha, isse zyada KABHI nahi banenge!)


## 🎯 Bonus Challenge

Ab try karo:
1. `max_size=2` rakhte hue, EK SAATH (bina return kiye) 3 workers borrow karne ki koshish karo —
   dekho `"Pool full hai!"` exception aata hai ya nahi. Ye samajhne mein help karega ki
   **limited resources** ko pool kaise CONTROL karta hai.
2. Ek `is_valid()` method add karo `Worker` class mein (jaise document ke Java example mein
   `validate()` tha) — jo check kare ki worker abhi bhi "healthy" hai ya nahi (e.g. random
   simulate karo). Pool ko `borrow_worker()` mein is check ko use karna chahiye, aur agar
   worker INVALID hai to use "expire" karke naya banana chahiye.
3. Socho: agar koi client `return_worker()` call karna HI BHOOL jaaye, to kya problem hogi?
   (Hint: ye document mein mention **"Risk of resource leakage"** disadvantage se connected hai)